# U.S. Recession Probability Model — 12-Month Ahead

A **probit regression** using NBER recession dates as the binary dependent variable,
driven by economic indicators across eight categories.

Based on the methodology of Estrella & Mishkin (1996, 1998), as used by the
NY Fed and Cleveland Fed.

**Framework:** P(Recession\_{t+12} = 1 | X\_t) = Φ(α₀ + α₁X\_t)

Where Φ(·) is the standard normal CDF and X\_t is a vector of economic indicators.

### Key design choices (all configurable below):
1. **Dependent variable**: "recession at month t+12" vs. "any recession in next 12 months"
2. **Feature selection**: data-driven via BIC, not hardcoded
3. **Estimation**: expanding-window pseudo out-of-sample
4. **Data universe**: all 41 FRED series across eight macroeconomic categories


In [ ]:
# Install dependencies
!pip install fredapi statsmodels scikit-learn matplotlib pandas numpy scipy -q

In [ ]:
# ============================================================
# FRED API Key Setup
# Get your free key at: https://fred.stlouisfed.org/docs/api/fred/
# ============================================================
from getpass import getpass

FRED_API_KEY = getpass("Enter your FRED API key: ")

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from fredapi import Fred
from scipy import stats
from sklearn.metrics import roc_auc_score, brier_score_loss
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

fred = Fred(api_key=FRED_API_KEY)
print("FRED connection established.")

## Configuration

All modeling choices are controlled here. Nothing downstream is hardcoded.

In [ ]:
# ============================================================
# MODEL CONFIGURATION — adjust these to explore alternatives
# ============================================================

# Dependent variable definition:
#   "point"  = recession specifically at month t+12 (NY Fed approach)
#   "window" = any recession during months t+1 through t+12
TARGET_DEFINITION = "point"

# Observation start date (CFNAI begins 1967)
OBS_START = "1967-01-01"

# Minimum expanding-window training size (months)
MIN_WINDOW = 120  # 10 years

# Maximum number of features for BIC selection
MAX_FEATURES_BIC = 9

# Threshold levels for probability interpretation
THRESHOLD_WARNING = 30   # "warrants attention"
THRESHOLD_ELEVATED = 50  # "more likely than not"

print(f"Target definition:  {TARGET_DEFINITION}")
print(f"Observation start:  {OBS_START}")
print(f"Min training window: {MIN_WINDOW} months")
print(f"Max BIC features:   {MAX_FEATURES_BIC}")

## Stage 1: Data Pipeline — All 41 FRED Series

The full indicator universe across eight macroeconomic categories.
The first series in each category is the strongest predictor per academic evidence.

In [ ]:
# ============================================================
# All 41 FRED series across 8 categories
# ============================================================
SERIES_CONFIG = {
    # --- National Economic Activity ---
    "CFNAI":    {"name": "Chicago Fed National Activity Index",   "category": "National Activity", "transform": "level"},
    "CFNAIMA3": {"name": "CFNAI 3-Month Moving Average",         "category": "National Activity", "transform": "level"},
    "GDPC1":    {"name": "Real GDP",                              "category": "National Activity", "transform": "yoy", "freq": "Q"},
    "USSLIND":  {"name": "Leading Index for the US",              "category": "National Activity", "transform": "level"},

    # --- Industrial Indicators ---
    "INDPRO":   {"name": "Industrial Production Index",           "category": "Industrial", "transform": "yoy"},
    "BSCICP02USM460S": {"name": "OECD Manufacturing Confidence",        "category": "Industrial", "transform": "level"},
    "TCU":      {"name": "Capacity Utilization",                  "category": "Industrial", "transform": "level"},
    "DGORDER":  {"name": "Durable Goods Orders",                  "category": "Industrial", "transform": "yoy"},
    "IPMAN":    {"name": "Industrial Production: Manufacturing",  "category": "Industrial", "transform": "yoy"},

    # --- Consumer Measures ---
    "UMCSENT":  {"name": "U. Michigan Consumer Sentiment",        "category": "Consumer", "transform": "level"},
    "PCECC96":  {"name": "Real Personal Consumption Expenditures","category": "Consumer", "transform": "yoy"},
    "DSPIC96":  {"name": "Real Disposable Personal Income",       "category": "Consumer", "transform": "yoy"},
    "RSAFS":    {"name": "Advance Retail Sales",                  "category": "Consumer", "transform": "yoy"},

    # --- Labor Market ---
    "UNRATE":   {"name": "Unemployment Rate",                     "category": "Labor", "transform": "level"},
    "ICSA":     {"name": "Initial Unemployment Claims",           "category": "Labor", "transform": "yoy", "freq": "W"},
    "PAYEMS":   {"name": "Total Nonfarm Payrolls",                "category": "Labor", "transform": "yoy"},
    "CIVPART":  {"name": "Labor Force Participation Rate",        "category": "Labor", "transform": "level"},
    "JTSJOL":   {"name": "Job Openings (JOLTS)",                  "category": "Labor", "transform": "yoy"},

    # --- Inflation ---
    "CPIAUCSL": {"name": "CPI All Urban Consumers",              "category": "Inflation", "transform": "yoy"},
    "PCEPILFE": {"name": "Core PCE Price Index",                  "category": "Inflation", "transform": "yoy"},
    "PCEPI":    {"name": "PCE Chain-Type Price Index",            "category": "Inflation", "transform": "yoy"},
    "CPILFESL": {"name": "Core CPI",                              "category": "Inflation", "transform": "yoy"},
    "PPIACO":   {"name": "PPI All Commodities",                   "category": "Inflation", "transform": "yoy"},

    # --- Housing ---
    "HOUST":    {"name": "Housing Starts",                        "category": "Housing", "transform": "yoy"},
    "PERMIT":   {"name": "Building Permits",                      "category": "Housing", "transform": "yoy"},
    "HSN1F":    {"name": "New One-Family Houses Sold",            "category": "Housing", "transform": "yoy"},
    "CSUSHPISA":{"name": "Case-Shiller National Home Price Index","category": "Housing", "transform": "yoy"},

    # --- Banking / Credit ---
    "BAA10YM":  {"name": "Baa Corp Bond - 10Y Treasury Spread",  "category": "Banking", "transform": "level"},
    "BUSLOANS": {"name": "Commercial & Industrial Loans",         "category": "Banking", "transform": "yoy"},
    "DRALACBS": {"name": "Delinquency Rate, All Loans",           "category": "Banking", "transform": "level", "freq": "Q"},
    "DRTSCILM": {"name": "Tightening Standards C&I Loans",        "category": "Banking", "transform": "level", "freq": "Q"},

    # --- Government Bond Yields ---
    "T10Y3M":   {"name": "10Y-3M Treasury Spread",               "category": "Yields", "transform": "level", "freq": "D"},
    "T10Y2Y":   {"name": "10Y-2Y Treasury Spread",               "category": "Yields", "transform": "level", "freq": "D"},
    "GS10":     {"name": "10-Year Treasury Yield",                "category": "Yields", "transform": "level"},
    "TB3MS":    {"name": "3-Month Treasury Bill Rate",            "category": "Yields", "transform": "level"},
    "FEDFUNDS": {"name": "Federal Funds Rate",                    "category": "Yields", "transform": "level"},
}

# Target variable (separate — not a feature)
TARGET_SERIES = {
    "USREC":         {"name": "NBER Recession Indicator",         "category": "Target"},
    "RECPROUSM156N": {"name": "Chauvet-Piger Recession Prob",     "category": "Benchmark"},
}

print(f"Data universe: {len(SERIES_CONFIG)} indicator series + {len(TARGET_SERIES)} target series")
print(f"Categories: {sorted(set(v['category'] for v in SERIES_CONFIG.values()))}")

In [ ]:
# ============================================================
# Fetch all series from FRED (with retry for transient errors)
# ============================================================
import time

OBS_START = "1967-01-01"  # CFNAI starts 1967

raw_data = pd.DataFrame()
failed = []

all_series = {**SERIES_CONFIG, **TARGET_SERIES}
MAX_RETRIES = 4

for sid, info in all_series.items():
    success = False
    for attempt in range(MAX_RETRIES):
        try:
            s = fred.get_series(sid, observation_start=OBS_START)

            # Resample non-monthly frequencies to monthly
            freq = info.get("freq", "M")
            if freq == "W":
                s = s.resample("MS").mean()
            elif freq == "D":
                s = s.resample("MS").last()
            elif freq == "Q":
                # Forward-fill quarterly to monthly
                s = s.resample("MS").ffill()

            raw_data[sid] = s
            print(f"  + {sid:12s} [{info.get('freq','M'):>1s}] {info['name']}")
            success = True
            break
        except Exception as e:
            if attempt < MAX_RETRIES - 1:
                wait = 2 ** (attempt + 1)  # 2s, 4s, 8s
                print(f"  ~ {sid:12s}       Retry {attempt+1}/{MAX_RETRIES-1} in {wait}s ({e})")
                time.sleep(wait)
            else:
                failed.append(sid)
                print(f"  x {sid:12s}       FAILED after {MAX_RETRIES} attempts: {e}")

# Ensure monthly alignment
raw_data.index = pd.to_datetime(raw_data.index)
raw_data = raw_data.resample("MS").last()

print(f"\nFetched {len(all_series) - len(failed)}/{len(all_series)} series.")
print(f"Date range: {raw_data.index.min().strftime('%Y-%m')} to {raw_data.index.max().strftime('%Y-%m')}")
if failed:
    print(f"Failed: {failed}")
    print("Note: FRED API can have transient errors. Re-run this cell to retry.")
raw_data.tail(3)

## Stage 2: Feature Engineering

Transformations are driven by the `transform` field in `SERIES_CONFIG`:
- `"yoy"` → 12-month percent change (for non-stationary level series)
- `"level"` → used as-is (for stationary/bounded series like spreads, rates, indexes)

Additionally constructs:
- **SPREAD** = GS10 − TB3MS (manual yield curve spread for pre-1982 history)
- **UNRATE_CHG3** = Sahm-style 3-month MA change in unemployment
- **Both dependent variable definitions** (configurable via `TARGET_DEFINITION`)

In [ ]:
# ============================================================
# Feature Engineering — transform-driven
# ============================================================
data = raw_data.copy()

# --- Construct derived spread (GS10 - TB3MS) for full history ---
if "GS10" in data.columns and "TB3MS" in data.columns:
    data["SPREAD"] = data["GS10"] - data["TB3MS"]

# --- Sahm-style unemployment rate change ---
if "UNRATE" in data.columns:
    unrate_ma3 = data["UNRATE"].rolling(3).mean()
    data["UNRATE_CHG3"] = unrate_ma3 - unrate_ma3.shift(12)

# --- Apply transforms from SERIES_CONFIG ---
feature_cols = []

for sid, info in SERIES_CONFIG.items():
    if sid not in data.columns:
        continue

    transform = info["transform"]

    if transform == "yoy":
        col_name = f"{sid}_YOY"
        data[col_name] = data[sid].pct_change(12) * 100
        feature_cols.append(col_name)
    elif transform == "level":
        feature_cols.append(sid)

# Add the derived features
if "SPREAD" in data.columns:
    feature_cols.append("SPREAD")
if "UNRATE_CHG3" in data.columns:
    feature_cols.append("UNRATE_CHG3")

# De-duplicate (SPREAD components GS10/TB3MS already in as levels)
feature_cols = sorted(set(feature_cols))

# --- Dependent variable: configurable ---
if TARGET_DEFINITION == "point":
    data["TARGET"] = data["USREC"].shift(-12)
    target_desc = "Recession specifically at month t+12"
elif TARGET_DEFINITION == "window":
    data["TARGET"] = data["USREC"].rolling(window=12).max().shift(-12)
    target_desc = "Any recession during months t+1 through t+12"
else:
    raise ValueError(f"Unknown TARGET_DEFINITION: {TARGET_DEFINITION}")

print(f"Target definition: {target_desc}")
print(f"Candidate features: {len(feature_cols)}")
print(f"Features: {feature_cols}")

In [ ]:
# ============================================================
# Build model-ready dataframe — drop rows with any NaN
# ============================================================
available_features = [c for c in feature_cols if c in data.columns]
model_df = data[available_features + ["TARGET", "USREC"]].dropna()

n_rec = int(model_df["TARGET"].sum())
pct_rec = model_df["TARGET"].mean() * 100

print(f"Model dataset: {len(model_df)} observations")
print(f"Date range:    {model_df.index.min().strftime('%Y-%m')} to {model_df.index.max().strftime('%Y-%m')}")
print(f"Recession obs: {n_rec} ({pct_rec:.1f}%)")
print(f"Features:      {len(available_features)}")

# Map features back to categories for reporting
feat_to_cat = {}
for sid, info in SERIES_CONFIG.items():
    cat = info["category"]
    if info["transform"] == "yoy":
        feat_to_cat[f"{sid}_YOY"] = cat
    else:
        feat_to_cat[sid] = cat
feat_to_cat["SPREAD"] = "Yields (derived)"
feat_to_cat["UNRATE_CHG3"] = "Labor (derived)"

print("\nFeatures by category:")
for cat in sorted(set(feat_to_cat.values())):
    feats = [f for f in available_features if feat_to_cat.get(f) == cat]
    if feats:
        print(f"  {cat}: {feats}")

## Exploratory Data Analysis

Visualize key indicators from each category against NBER recession periods.

In [ ]:
# ============================================================
# EDA: One indicator per category vs. recessions
# ============================================================
# Pick the top indicator per category for visualization
eda_picks = []
category_order = ["Yields", "Yields (derived)", "Banking", "National Activity",
                  "Industrial", "Consumer", "Labor", "Labor (derived)",
                  "Inflation", "Housing"]
seen_cats = set()
for cat in category_order:
    for f in available_features:
        if feat_to_cat.get(f) == cat and cat not in seen_cats:
            eda_picks.append((f, cat))
            seen_cats.add(cat)
            break

n_plots = min(len(eda_picks), 10)
rows = (n_plots + 1) // 2
fig, axes = plt.subplots(rows, 2, figsize=(16, 3.2 * rows), sharex=True)
axes = axes.flat

usrec = data["USREC"].dropna()

for i, (col, cat) in enumerate(eda_picks[:n_plots]):
    ax = axes[i]
    if col in data.columns:
        s = data[col].dropna()
        ax.plot(s.index, s.values, linewidth=0.9, color="#1f77b4")
    ax.fill_between(usrec.index, ax.get_ylim()[0], ax.get_ylim()[1],
                    where=usrec.values == 1, color="gray", alpha=0.2)
    ax.set_title(f"{col}  [{cat}]", fontsize=10)
    ax.grid(True, alpha=0.15)

# Hide unused subplots
for j in range(n_plots, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Key Indicators by Category vs. NBER Recessions", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Stage 3: Data-Driven Feature Selection via BIC

Instead of hardcoding features, we use **forward stepwise selection by BIC**
(Bayesian Information Criterion) to let the data determine which indicators
enter the model.

BIC penalizes model complexity more heavily than AIC, favoring parsimony —
consistent with Berge (2014)'s finding that "at the 12-month horizon,
parsimony dominates."

The selection proceeds:
1. Start with SPREAD (the single strongest predictor per Estrella & Mishkin)
2. Greedily add the feature that produces the largest BIC improvement
3. Stop when no addition improves BIC, or `MAX_FEATURES_BIC` is reached

In [ ]:
# ============================================================
# Univariate screening: rank all features by individual BIC
# ============================================================
y = model_df["TARGET"].astype(float)

univariate_results = []

for feat in available_features:
    X = sm.add_constant(model_df[[feat]].astype(float))
    try:
        res = sm.Probit(y, X).fit(disp=False, method="bfgs", maxiter=300)
        univariate_results.append({
            "feature": feat,
            "category": feat_to_cat.get(feat, "?"),
            "bic": res.bic,
            "aic": res.aic,
            "pseudo_r2": res.prsquared,
            "coef": res.params.iloc[-1],
            "pvalue": res.pvalues.iloc[-1],
        })
    except Exception as e:
        print(f"  Skip {feat}: {e}")

uni_df = pd.DataFrame(univariate_results).sort_values("bic")
print("Univariate BIC ranking (lower = better):\n")
print(uni_df.to_string(index=False, float_format=lambda x: f"{x:.2f}"))

In [ ]:
# ============================================================
# Forward stepwise selection by BIC
# ============================================================
def forward_stepwise_bic(y, X_all, feature_names, max_features, seed_features=None):
    """
    Greedy forward selection. Starts with seed_features (if any),
    then adds features one at a time, choosing the one that most
    reduces BIC. Stops when BIC stops improving or max_features reached.
    """
    selected = list(seed_features) if seed_features else []
    remaining = [f for f in feature_names if f not in selected]

    # Fit seed model (or intercept-only)
    if selected:
        X_curr = sm.add_constant(X_all[selected].astype(float))
        best_bic = sm.Probit(y, X_curr).fit(disp=False, method="bfgs", maxiter=300).bic
    else:
        X_curr = sm.add_constant(pd.DataFrame(index=X_all.index))
        best_bic = sm.Probit(y, X_curr).fit(disp=False, method="bfgs", maxiter=300).bic

    history = [{"step": 0, "added": "/".join(selected) if selected else "(none)",
                "bic": best_bic, "n_features": len(selected)}]

    while remaining and len(selected) < max_features:
        candidates = []
        for feat in remaining:
            try_feats = selected + [feat]
            X_try = sm.add_constant(X_all[try_feats].astype(float))
            try:
                res = sm.Probit(y, X_try).fit(disp=False, method="bfgs", maxiter=300)
                candidates.append((feat, res.bic))
            except Exception:
                pass

        if not candidates:
            break

        best_feat, best_candidate_bic = min(candidates, key=lambda x: x[1])

        if best_candidate_bic >= best_bic:
            # No improvement — stop
            break

        selected.append(best_feat)
        remaining.remove(best_feat)
        best_bic = best_candidate_bic

        history.append({"step": len(selected), "added": best_feat,
                        "bic": best_bic, "n_features": len(selected)})
        print(f"  Step {len(selected):2d}: +{best_feat:<18s} BIC={best_bic:.2f}")

    return selected, history


# Determine seed: use SPREAD if available (strongest single predictor)
seed = ["SPREAD"] if "SPREAD" in available_features else []

print(f"Seed features: {seed if seed else '(none)'}")
print(f"Candidate pool: {len(available_features)} features")
print(f"Max features: {MAX_FEATURES_BIC}\n")

bic_selected, bic_history = forward_stepwise_bic(
    y=y,
    X_all=model_df[available_features],
    feature_names=available_features,
    max_features=MAX_FEATURES_BIC,
    seed_features=seed,
)

print(f"\nBIC-selected features ({len(bic_selected)}):")
for i, f in enumerate(bic_selected):
    cat = feat_to_cat.get(f, "?")
    print(f"  {i+1}. {f:<20s} [{cat}]")

In [ ]:
# ============================================================
# BIC selection path visualization
# ============================================================
hist_df = pd.DataFrame(bic_history)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hist_df["n_features"], hist_df["bic"], "o-", color="#1f77b4", linewidth=2, markersize=8)
for _, row in hist_df.iterrows():
    ax.annotate(row["added"], (row["n_features"], row["bic"]),
                textcoords="offset points", xytext=(8, 8), fontsize=8, rotation=20)
ax.set_xlabel("Number of Features")
ax.set_ylabel("BIC (lower = better)")
ax.set_title("Forward Stepwise BIC Selection Path", fontweight="bold")
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

## Stage 4: Probit Model Estimation

Four models of increasing complexity:
1. **NY Fed baseline**: Yield curve spread only (Estrella & Mishkin 1998)
2. **Wright extension**: Spread + Federal Funds Rate level (Wright 2006)
3. **BIC-selected**: Data-driven feature set from stepwise selection
4. **Full candidate set**: All available features (likely overfits — included for comparison)

All use `statsmodels.Probit` with BFGS optimization.

In [ ]:
# ============================================================
# Define model specifications
# ============================================================

# Build Wright features from available data
wright_features = []
if "SPREAD" in available_features:
    wright_features.append("SPREAD")
elif "T10Y3M" in available_features:
    wright_features.append("T10Y3M")
if "FEDFUNDS" in available_features:
    wright_features.append("FEDFUNDS")

# NY Fed: spread only
nyfed_features = [wright_features[0]] if wright_features else [available_features[0]]

models_spec = {
    "NY Fed (Spread Only)": nyfed_features,
    "Wright (Spread + FF)": wright_features,
    "BIC-Selected":         bic_selected,
    "Full Candidate Set":   available_features,
}

results = {}

for name, features in models_spec.items():
    feats = [f for f in features if f in model_df.columns]
    if not feats:
        print(f"  {name}: no valid features, skipping")
        continue

    X = sm.add_constant(model_df[feats].astype(float))
    try:
        res = sm.Probit(y, X).fit(disp=False, method="bfgs", maxiter=500)
        fitted = res.predict(X)
        results[name] = {
            "model": res,
            "features": feats,
            "fitted": fitted,
            "pseudo_r2": res.prsquared,
            "bic": res.bic,
            "aic": res.aic,
        }
        print(f"{'='*60}")
        print(f"  {name}  ({len(feats)} features)")
        print(f"{'='*60}")
        print(f"  Pseudo R²:  {res.prsquared:.4f}")
        print(f"  Log-Lik:    {res.llf:.1f}")
        print(f"  AIC:        {res.aic:.1f}")
        print(f"  BIC:        {res.bic:.1f}")
        print(f"  Features:   {feats}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")

print("\n" + "="*60)
print("  Model estimation complete.")

In [ ]:
# ============================================================
# BIC-selected model — full coefficient table
# ============================================================
if "BIC-Selected" in results:
    print("BIC-Selected Model Summary:\n")
    print(results["BIC-Selected"]["model"].summary())
else:
    print("BIC-Selected model not available.")

## Stage 4b: Expanding-Window Out-of-Sample Estimation

Each month's probability is generated from a model trained **only on data
available at that point**. This is the honest evaluation — no lookahead bias.

Uses the BIC-selected features (data-driven, not hardcoded).

In [ ]:
# ============================================================
# Expanding-window pseudo out-of-sample — BIC-selected model
# ============================================================
oos_features = bic_selected  # data-driven, not hardcoded
oos_probs = pd.Series(index=model_df.index, dtype=float)

n = len(model_df)
total_iters = n - MIN_WINDOW
print(f"Running expanding-window OOS estimation ({total_iters} iterations)...")
print(f"Features: {oos_features}\n")

for i in range(MIN_WINDOW, n):
    train = model_df.iloc[:i]
    y_t = train["TARGET"].astype(float)
    X_t = sm.add_constant(train[oos_features].astype(float))

    try:
        res = sm.Probit(y_t, X_t).fit(disp=False, method="bfgs", maxiter=300)
        X_curr = sm.add_constant(
            model_df[oos_features].iloc[[i]].astype(float)
        )
        oos_probs.iloc[i] = res.predict(X_curr)[0]
    except Exception:
        oos_probs.iloc[i] = float("nan")

    if (i - MIN_WINDOW) % 100 == 0:
        pct = (i - MIN_WINDOW) / total_iters * 100
        print(f"  {pct:5.1f}% complete (month {model_df.index[i].strftime('%Y-%m')})")

print(f"\nOut-of-sample estimation complete.")
print(f"Valid OOS probabilities: {oos_probs.notna().sum()}")

## Model Evaluation

Compare all four models using AUROC, Brier Score, AIC, BIC, and Pseudo R².

The BIC-selected model should dominate on BIC by construction. The key
question is whether it also dominates on AUROC out-of-sample.

In [ ]:
# ============================================================
# Evaluation metrics — all models
# ============================================================
y_true = model_df["TARGET"].astype(float)

header = f"{'Model':<28s} {'AUROC':<10s} {'Brier':<10s} {'Pseudo R²':<10s} {'AIC':<10s} {'BIC':<10s} {'# Feat':<6s}"
print(header)
print("-" * len(header))

for name, res_dict in results.items():
    fitted = res_dict["fitted"]
    auroc = roc_auc_score(y_true, fitted)
    brier = brier_score_loss(y_true, fitted)
    pr2 = res_dict["pseudo_r2"]
    aic = res_dict["aic"]
    bic = res_dict["bic"]
    nf = len(res_dict["features"])
    print(f"{name:<28s} {auroc:<10.4f} {brier:<10.4f} {pr2:<10.4f} {aic:<10.1f} {bic:<10.1f} {nf:<6d}")

# OOS metrics for BIC-selected
oos_valid = oos_probs.dropna()
if len(oos_valid) > 0:
    y_oos = model_df.loc[oos_valid.index, "TARGET"].astype(float)
    auroc_oos = roc_auc_score(y_oos, oos_valid)
    brier_oos = brier_score_loss(y_oos, oos_valid)
    print(f"{'BIC-Selected (OOS)':<28s} {auroc_oos:<10.4f} {brier_oos:<10.4f} {'—':<10s} {'—':<10s} {'—':<10s} {len(bic_selected):<6d}")

## Dependent Variable Comparison: Point-in-Time vs. Any-in-Window

The Boston Fed (2020) found "considerable dispersion in predicted recession
probabilities" depending on this choice. Here we fit the BIC-selected model
under **both** definitions and compare.

In [ ]:
# ============================================================
# Compare both dependent variable definitions
# ============================================================
target_point  = data["USREC"].shift(-12)
target_window = data["USREC"].rolling(window=12).max().shift(-12)

dv_results = {}

for dv_name, dv_series in [("Point (t+12)", target_point),
                            ("Window (any in 1-12)", target_window)]:
    # Align with model_df index
    dv = dv_series.reindex(model_df.index).dropna()
    common_idx = dv.index.intersection(model_df.index)
    y_dv = dv.loc[common_idx].astype(float)
    X_dv = sm.add_constant(model_df.loc[common_idx, bic_selected].astype(float))

    try:
        res = sm.Probit(y_dv, X_dv).fit(disp=False, method="bfgs", maxiter=500)
        fitted = res.predict(X_dv)
        auroc = roc_auc_score(y_dv, fitted)
        brier = brier_score_loss(y_dv, fitted)
        dv_results[dv_name] = {
            "fitted": fitted,
            "y": y_dv,
            "pseudo_r2": res.prsquared,
            "auroc": auroc,
            "brier": brier,
        }
        pct_pos = y_dv.mean() * 100
        print(f"{dv_name}:")
        print(f"  Positive rate: {pct_pos:.1f}%   Pseudo R²: {res.prsquared:.4f}   AUROC: {auroc:.4f}   Brier: {brier:.4f}")
    except Exception as e:
        print(f"{dv_name}: FAILED — {e}")

In [ ]:
# ============================================================
# Side-by-side chart: both DV definitions
# ============================================================
if len(dv_results) == 2:
    fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
    usrec = data["USREC"].dropna()

    for ax, (dv_name, dv_dict) in zip(axes, dv_results.items()):
        ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                        color="#d4d4d4", alpha=0.6, label="NBER Recession")
        fitted = dv_dict["fitted"]
        ax.plot(fitted.index, fitted * 100, color="#1f77b4", linewidth=1.2)
        ax.axhline(y=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
        ax.axhline(y=THRESHOLD_WARNING, color="orange", linestyle=":", alpha=0.3, linewidth=0.8)
        ax.set_ylim(0, 100)
        ax.set_ylabel("Probability (%)")
        r2 = dv_dict["pseudo_r2"]
        auroc = dv_dict["auroc"]
        ax.set_title(f"{dv_name}  (Pseudo R²={r2:.4f}, AUROC={auroc:.4f})", fontweight="bold")
        ax.legend(loc="upper right", fontsize=9)
        ax.grid(True, alpha=0.15)

    axes[0].xaxis.set_major_locator(mdates.YearLocator(5))
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    plt.suptitle("Dependent Variable Comparison — BIC-Selected Model", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Could not fit both DV definitions.")

## Stage 5: Visualization

Main chart: 12-month-ahead recession probability with NBER recession shading.
Shows both in-sample (BIC-selected) and out-of-sample probabilities.

In [ ]:
# ============================================================
# Main Recession Probability Chart
# ============================================================
fig, ax = plt.subplots(figsize=(16, 6))

# NBER recession shading
usrec = data["USREC"].dropna()
ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                color="#d4d4d4", alpha=0.6, label="NBER Recession")

# In-sample (BIC-selected)
if "BIC-Selected" in results:
    fitted = results["BIC-Selected"]["fitted"]
    ax.plot(fitted.index, fitted * 100,
            color="#1f77b4", linewidth=1.2, alpha=0.5,
            label="In-Sample (BIC-Selected)")

# Out-of-sample
oos_valid = oos_probs.dropna()
if len(oos_valid) > 0:
    ax.plot(oos_valid.index, oos_valid * 100,
            color="#d62728", linewidth=1.5,
            label="Out-of-Sample (Expanding Window)")

# Threshold lines
ax.axhline(y=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.3,
           linewidth=0.8, label=f"{THRESHOLD_ELEVATED}% Threshold")
ax.axhline(y=THRESHOLD_WARNING, color="orange", linestyle=":", alpha=0.3,
           linewidth=0.8, label=f"{THRESHOLD_WARNING}% Warning")

# Styling
ax.set_ylim(0, 100)
ax.set_xlim(model_df.index.min(), data.index.max())
ax.set_ylabel("Probability (%)", fontsize=12)
dv_label = "Point-in-Time" if TARGET_DEFINITION == "point" else "Any-in-Window"
ax.set_title(f"U.S. Recession Probability — 12-Month Ahead ({dv_label}, BIC-Selected Probit)",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("recession_probability_main.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved recession_probability_main.png")

## Model Comparison Chart

All four specifications overlaid: NY Fed baseline, Wright, BIC-selected, and full candidate set.

In [ ]:
# ============================================================
# Model comparison chart
# ============================================================
palette = ["#2ca02c", "#ff7f0e", "#1f77b4", "#9467bd"]
fig, ax = plt.subplots(figsize=(16, 6))

usrec = data["USREC"].dropna()
ax.fill_between(usrec.index, 0, 100, where=usrec.values == 1,
                color="#d4d4d4", alpha=0.6, label="NBER Recession")

for (name, res_dict), color in zip(results.items(), palette):
    fitted = res_dict["fitted"]
    r2 = res_dict["pseudo_r2"]
    bic_val = res_dict["bic"]
    ax.plot(fitted.index, fitted * 100, color=color,
            linewidth=1.0, alpha=0.8,
            label=f"{name} (R²={r2:.3f}, BIC={bic_val:.0f})")

ax.axhline(y=THRESHOLD_ELEVATED, color="red", linestyle="--", alpha=0.3, linewidth=0.8)
ax.set_ylim(0, 100)
ax.set_ylabel("Probability (%)", fontsize=12)
ax.set_title("Model Comparison — 12-Month-Ahead Recession Probability", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=8)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.grid(True, alpha=0.15)

plt.tight_layout()
plt.savefig("recession_probability_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## Current Recession Probability Reading

In [ ]:
# ============================================================
# Current reading — all models
# ============================================================
print("=" * 65)
print("  CURRENT 12-MONTH-AHEAD RECESSION PROBABILITY")
print("=" * 65)

for name, res_dict in results.items():
    p = res_dict["fitted"].iloc[-1] * 100
    date = res_dict["fitted"].index[-1].strftime('%B %Y')
    nf = len(res_dict["features"])
    print(f"  {name:<28s}: {p:5.1f}%   (as of {date}, {nf} features)")

print()

# Latest from BIC-selected
if "BIC-Selected" in results:
    latest_prob = results["BIC-Selected"]["fitted"].iloc[-1] * 100
    if latest_prob > THRESHOLD_ELEVATED:
        print(f"  ELEVATED: BIC-Selected probability ({latest_prob:.1f}%) exceeds {THRESHOLD_ELEVATED}%")
        print(f"  — recession more likely than not within 12 months.")
    elif latest_prob > THRESHOLD_WARNING:
        print(f"  WARNING: BIC-Selected probability ({latest_prob:.1f}%) above {THRESHOLD_WARNING}%")
        print(f"  — warrants attention.")
    else:
        print(f"  LOW: BIC-Selected probability ({latest_prob:.1f}%) below {THRESHOLD_WARNING}%")
        print(f"  — recession risk contained.")

    print("\n  Current indicator readings (BIC-selected features):")
    latest_row = model_df[bic_selected].iloc[-1]
    for feat in bic_selected:
        cat = feat_to_cat.get(feat, "?")
        print(f"    {feat:<20s} = {latest_row[feat]:>8.2f}   [{cat}]")

## Estrella-Mishkin Quick Estimate

Using pre-estimated parameters from Estrella & Trubin (2006):
**P(recession) = Φ(−0.6045 − 0.7374 × spread)**

No model fitting required — just plug in the current spread.

In [ ]:
# ============================================================
# Estrella-Mishkin closed-form estimate
# ============================================================
spread_col = "SPREAD" if "SPREAD" in data.columns else "T10Y3M" if "T10Y3M" in data.columns else None

if spread_col:
    spread_latest = data[spread_col].dropna().iloc[-1]
    em_prob = stats.norm.cdf(-0.6045 - 0.7374 * spread_latest) * 100

    print(f"Current 10Y-3M spread:      {spread_latest:.2f}%")
    print(f"Estrella-Mishkin 12m prob:   {em_prob:.1f}%")
    print()

    # Plot the mapping curve
    spread_range = np.linspace(-4, 5, 200)
    em_curve = stats.norm.cdf(-0.6045 - 0.7374 * spread_range) * 100

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(spread_range, em_curve, "b-", linewidth=2)
    ax.axvline(x=spread_latest, color="red", linestyle="--", alpha=0.6,
               label=f"Current spread: {spread_latest:.2f}%")
    ax.axvline(x=0, color="gray", linestyle=":", alpha=0.4, label="Inversion point")
    ax.scatter([spread_latest], [em_prob], color="red", s=80, zorder=5)
    ax.annotate(f"{em_prob:.1f}%", (spread_latest, em_prob),
                textcoords="offset points", xytext=(15, 10), fontsize=11,
                arrowprops=dict(arrowstyle="->", color="red"))
    ax.set_xlabel("10Y-3M Treasury Spread (%)", fontsize=12)
    ax.set_ylabel("Recession Probability (%)", fontsize=12)
    ax.set_title("Estrella-Mishkin Yield Curve Model", fontsize=14, fontweight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.15)
    ax.set_ylim(0, 100)
    plt.tight_layout()
    plt.show()
else:
    print("No spread series available for Estrella-Mishkin estimate.")

## Feature Importance — BIC-Selected Model

z-statistics and marginal effects for the BIC-selected features.
Red bars = statistically significant at 5%; blue = not significant.

In [ ]:
# ============================================================
# Feature importance: z-statistics and marginal effects
# ============================================================
if "BIC-Selected" in results:
    res = results["BIC-Selected"]["model"]

    coef_df = pd.DataFrame({
        "Coefficient": res.params,
        "Std Error": res.bse,
        "z-stat": res.tvalues,
        "p-value": res.pvalues,
        "|z-stat|": res.tvalues.abs(),
    }).drop("const", errors="ignore")

    coef_df = coef_df.sort_values("|z-stat|", ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(coef_df) * 0.5)))

    # Panel 1: z-statistics
    colors_z = ["#d62728" if p < 0.05 else "#aec7e8" for p in coef_df["p-value"]]
    axes[0].barh(coef_df.index, coef_df["|z-stat|"], color=colors_z)
    axes[0].axvline(x=1.96, color="gray", linestyle="--", alpha=0.5, label="5% significance")
    axes[0].set_xlabel("|z-statistic|")
    axes[0].set_title("Statistical Significance of Predictors")
    axes[0].legend()

    # Panel 2: Marginal effects at the mean
    mfx = res.get_margeff(at="mean")
    mfx_df = pd.DataFrame({
        "Marginal Effect": mfx.margeff,
        "Feature": list(coef_df.index),
    }).set_index("Feature").sort_values("Marginal Effect")

    mfx_colors = ["#d62728" if v > 0 else "#2ca02c" for v in mfx_df["Marginal Effect"]]
    axes[1].barh(mfx_df.index, mfx_df["Marginal Effect"], color=mfx_colors)
    axes[1].axvline(x=0, color="gray", linewidth=0.8)
    axes[1].set_xlabel("Marginal Effect on P(Recession)")
    axes[1].set_title("Marginal Effects at the Mean")

    plt.suptitle("Feature Importance — BIC-Selected Probit",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

## Correlation Matrix — BIC-Selected Features

In [ ]:
# ============================================================
# Correlation heatmap of BIC-selected features
# ============================================================
corr_cols = [f for f in bic_selected if f in model_df.columns]
corr_matrix = model_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(max(6, len(corr_cols)), max(5, len(corr_cols) * 0.8)))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)

for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=7,
                color="white" if abs(corr_matrix.iloc[i, j]) > 0.6 else "black")

plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("BIC-Selected Feature Correlation Matrix", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Historical Recession Detection Performance

How well did each model detect historical recessions 12 months in advance?
Checks peak probability in the 6–18 month window prior to each recession start.

In [ ]:
# ============================================================
# Historical recession detection — all models
# ============================================================
usrec_model = model_df["USREC"]
rec_starts = usrec_model[(usrec_model == 1) & (usrec_model.shift(1) == 0)].index

header = f"{'Recession':<14s}"
for name in results:
    short = name.split("(")[0].strip()[:12]
    header += f" {short:>14s}"
print(header)
print("-" * len(header))

for start in rec_starts:
    row = f"  {start.strftime('%Y-%m'):<12s}"
    for name, res_dict in results.items():
        fitted = res_dict["fitted"]
        window_start = start - pd.DateOffset(months=18)
        window_end = start - pd.DateOffset(months=6)
        window_probs = fitted.loc[window_start:window_end]
        if len(window_probs) > 0:
            peak = window_probs.max() * 100
            marker = " *" if peak > THRESHOLD_WARNING else ""
            row += f" {peak:>12.1f}%{marker}"
        else:
            row += f" {'—':>14s}"
    print(row)

print(f"\n  * = peak probability exceeded {THRESHOLD_WARNING}% warning threshold")

## Export Results

Save all probabilities, indicator data, and model metadata to CSV.

In [ ]:
# ============================================================
# Export to CSV
# ============================================================
export_df = pd.DataFrame(index=model_df.index)
export_df["USREC"] = model_df["USREC"]
export_df["TARGET_actual"] = model_df["TARGET"]
export_df["target_definition"] = TARGET_DEFINITION

for name, res_dict in results.items():
    col_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("-","_")
    export_df[f"Prob_{col_name}"] = res_dict["fitted"] * 100

export_df["Prob_OOS_BIC_Selected"] = oos_probs * 100

# Add all BIC-selected indicators
for feat in bic_selected:
    if feat in model_df.columns:
        export_df[feat] = model_df[feat]

export_df.to_csv("recession_probabilities.csv")
print(f"Exported {len(export_df)} rows x {len(export_df.columns)} columns to recession_probabilities.csv")
print(f"Columns: {list(export_df.columns)}")
export_df.tail()

## References

1. **Estrella, A. & Mishkin, F.S. (1998)**. "Predicting U.S. Recessions: Financial Variables as Leading Indicators." *Review of Economics and Statistics*, 80(1), 45-61.
2. **Wright, J.H. (2006)**. "The Yield Curve and Predicting Recessions." *Federal Reserve Board FEDS Working Paper* No. 2006-07.
3. **Kauppi, H. & Saikkonen, P. (2008)**. "Predicting U.S. Recessions with Dynamic Binary Response Models." *Review of Economics and Statistics*, 90(4), 777-791.
4. **Berge, T.J. (2014)**. "Predicting Recessions with Leading Indicators: Model Averaging and Links to the Financial Crisis." *Federal Reserve Bank of Kansas City Working Paper*.
5. **Federal Reserve Board FEDS Notes (2018, 2019)**. Various notes on recession probability models.
6. **Sahm, C. (2019)**. "Direct Stimulus Payments to Individuals." *Brookings Institution*.
7. **McCracken, M.W. & Ng, S. (2016)**. "FRED-MD: A Monthly Database for Macroeconomic Research." *Journal of Business & Economic Statistics*, 34(4), 574-589.
8. **Boston Fed (2020)**. On dispersion in recession probabilities from dependent variable construction.
9. **Berge, T.J. & Jordà, Ò. (2011)**. "Evaluating the Classification of Economic Activity into Recessions and Expansions." *American Economic Journal: Macroeconomics*.
10. **Bellego, C. & Ferrara, L. (2009)**. "Forecasting Euro Area Recessions Using Time-Varying Binary Response Models for Financial Variables." *ECB Working Paper*.

---

*Data sourced from FRED (Federal Reserve Bank of St. Louis).*
*Feature selection: forward stepwise by BIC. Dependent variable: configurable.*